In [ ]:
# 📦 Könyvtárak
import os, shutil, zipfile, random
from tqdm import tqdm
from glob import glob

# 🗂️ 1. Drive csatolása
from google.colab import drive
drive.mount('/content/drive')

# 🎯 2. Elérési utak
drive_zip_path = '/content/drive/MyDrive/diplomamunka/sorted_db.zip'
vm_zip_path = '/content/sorted_db.zip'
extracted_path = '/content/sorted_db'
splitted_path = '/content/splitted_db'

# 📥 3. ZIP fájl másolása Drive-ból a Colab VM-re
print("ZIP fájl másolása...")
shutil.copyfile(drive_zip_path, vm_zip_path)

# 📂 4. Kicsomagolás
print("Kicsomagolás...")
with zipfile.ZipFile(vm_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extracted_path)

# 🧪 5. Adathalmaz darabolása: train / val / test (80/10/10)
print("Adatok szétosztása train/val/test mappákba...")
splits = ['train', 'val', 'test']
ratios = [0.8, 0.1, 0.1]
os.makedirs(splitted_path, exist_ok=True)

# Minden kategóriát külön kezelünk
categories = [d for d in os.listdir(extracted_path) if os.path.isdir(os.path.join(extracted_path, d))]

for category in tqdm(categories, desc="📁 Kategóriák feldolgozása"):
    src_folder = os.path.join(extracted_path, category)
    images = glob(os.path.join(src_folder, '*'))
    random.shuffle(images)

    n_total = len(images)
    n_train = int(n_total * ratios[0])
    n_val = int(n_total * ratios[1])
    n_test = n_total - n_train - n_val

    split_counts = [n_train, n_val, n_test]
    split_names = zip(splits, split_counts)
    index = 0

    for split, count in split_names:
        split_folder = os.path.join(splitted_path, split, category)
        os.makedirs(split_folder, exist_ok=True)
        for img_path in images[index:index+count]:
            shutil.copy(img_path, os.path.join(split_folder, os.path.basename(img_path)))
        index += count

# 📦 6. Tömörítés splitted_db.zip-be
print("Tömörítés splitted_db.zip fájlba...")

def zipdir(path, ziph):
    for root, _, files in os.walk(path):
        for file in files:
            ziph.write(os.path.join(root, file),
                       os.path.relpath(os.path.join(root, file), path))

output_zip_path = '/content/splitted_db.zip'
with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipdir(splitted_path, zipf)

print("Kész! Letölthető fájl: splitted_db.zip ✅")
